In [ ]:
# Install required libraries for the lab.
%pip install openai cohere numpy

import os
import numpy as np
from openai import OpenAI
import cohere

# Replace these with your real API keys before running.
OPENAI_API_KEY = ""
COHERE_API_KEY = ""

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["COHERE_API_KEY"] = COHERE_API_KEY


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Phase 3: Dense Retrieval Limits & Cross-Encoder Reranking

This phase shows how dense retrieval can surface partially relevant but misleading results when queries involve negation, ambiguity, or overlapping keywords. It then applies cross-encoder reranking to the top candidates in order to improve the final ordering.

## Step 1: Create Document Dataset

In [9]:
# Build a 50-document dataset with negation, ambiguity, overlap, and normal examples.
negation_documents = [
    "Python is not suitable for mobile apps because most deployments target desktop or server workflows.",
    "This tutorial is not about Python programming for Android development.",
    "Mobile developers do not usually pick Python as their first choice for native iPhone apps.",
    "Python frameworks are not the standard route for building high performance mobile interfaces.",
    "The article explains why Python is not commonly used for native mobile game engines.",
    "Kotlin is preferred when the goal is not cross-platform scripting but true Android app development.",
    "Swift is not replaced by Python in most professional iOS mobile projects.",
    "This course does not recommend Python for beginners who only want native mobile programming.",
    "Python does not dominate the mobile app market the way it does in data science.",
    "BeeWare is interesting, but Python is not yet the default option for mobile development teams.",
]

overlapping_keyword_documents = [
    "Python is a large snake often found in tropical forests and wetlands.",
    "Apple released a new laptop while apple growers worried about frost damage.",
    "Java is an island in Indonesia, while Java is also a programming language.",
    "Windows let sunlight into the room, but Windows also dominates many office computers.",
    "Mobile networks rely on cell towers, while cell biology studies living organisms.",
    "Bugs in software frustrate engineers, but bugs in the garden can damage tomato plants.",
    "Threads can refer to online conversations or to fibers used in sewing.",
    "The mouse ate grain in the pantry, while the mouse on the desk controls the cursor.",
    "A kernel can mean the core of an operating system or the edible center of corn.",
    "Clouds brought rain to the farm, but cloud platforms host applications on remote servers.",
]

ambiguous_documents = [
    "The bank was covered in mud after the river overflowed during the storm.",
    "He saw the crane near the construction site just before it lifted steel beams.",
    "The bat flew out of the cave before the player picked up his bat for practice.",
    "They left the match on the table after the match ended in a draw.",
    "The coach watched the train coach roll past the station.",
    "She stood by the spring and wondered whether spring would arrive early this year.",
    "The seal rested on the rock beside the official seal on the shipping crate.",
    "He booked the court after reading the court report in the newspaper.",
    "The pitcher placed the pitcher on the bench after the final inning.",
    "The park ranger pointed to the rock band poster lying on a rock.",
]

normal_documents = [
    "BeeWare lets developers build desktop and mobile apps in Python from one codebase.",
    "Kivy is a Python framework that can target Android and iOS devices.",
    "Flutter is a popular toolkit for cross-platform mobile development using Dart.",
    "React Native allows JavaScript developers to create mobile apps for Android and iPhone.",
    "Native Android development typically uses Kotlin with Android Studio.",
    "Swift and SwiftUI are central tools for building modern iPhone applications.",
    "A healthy breakfast with fruit, oats, and yogurt can improve morning focus.",
    "Mountain hiking requires water, layered clothing, and awareness of trail conditions.",
    "Machine learning models often require careful preprocessing and evaluation.",
    "Electric vehicles reduce tailpipe emissions but still depend on battery supply chains.",
    "The history museum opened a new exhibit about ancient trade routes.",
    "Learning guitar takes patience, repetition, and attention to finger placement.",
    "Space telescopes capture data that helps astronomers study distant galaxies.",
    "Good passwords should be long, unique, and protected with multi-factor authentication.",
    "A well-designed garden uses soil quality, sunlight, and seasonal planning.",
    "Data analysts often use Python to automate reports and clean spreadsheets.",
    "Mobile accessibility features improve usability for people with visual or motor impairments.",
    "Progressive web apps can provide an app-like experience in a mobile browser.",
    "Battery optimization is important for mobile devices running background processes.",
    "Cross-platform app teams often evaluate performance tradeoffs before choosing a framework.",
]

documents = negation_documents + overlapping_keyword_documents + ambiguous_documents + normal_documents
print(f"Total documents: {len(documents)}")


Total documents: 50


## Step 2: Generate Embeddings

In [10]:
# Embed all 50 documents with the raw OpenAI SDK.
openai_client = OpenAI(api_key=OPENAI_API_KEY)
document_embedding_response = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=documents,
)

document_embeddings = np.array([item.embedding for item in document_embedding_response.data], dtype=np.float32)
print(f"Document embedding array shape: {document_embeddings.shape}")


Document embedding array shape: (50, 1536)


## Step 3: Define Query and Embed It

In [11]:
# Embed the query with the same model used for the documents.
query = "Python programming for mobile development"
query_embedding_response = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=[query],
)

query_embedding = np.array(query_embedding_response.data[0].embedding, dtype=np.float32)
print("Query embedded successfully.")


Query embedded successfully.


## Step 4: Stage 1 Dense Retrieval

In [12]:
# Run dense retrieval with manual cosine similarity.
def cosine_similarity_manual(vector_a, vector_b):
    numerator = np.dot(vector_a, vector_b)
    denominator = np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    return numerator / denominator

similarity_scores = np.array([
    cosine_similarity_manual(query_embedding, document_embedding)
    for document_embedding in document_embeddings
])

top_10_indices = np.argsort(-similarity_scores)[:10]
top_10_documents = [documents[index] for index in top_10_indices]

print("BEFORE RERANKING")
for rank, index in enumerate(top_10_indices, start=1):
    print(f"{rank}. Score={similarity_scores[index]:.4f} | {documents[index]}")


BEFORE RERANKING
1. Score=0.6693 | Python is not suitable for mobile apps because most deployments target desktop or server workflows.
2. Score=0.6076 | Python frameworks are not the standard route for building high performance mobile interfaces.
3. Score=0.5633 | Python does not dominate the mobile app market the way it does in data science.
4. Score=0.5606 | BeeWare is interesting, but Python is not yet the default option for mobile development teams.
5. Score=0.5449 | The article explains why Python is not commonly used for native mobile game engines.
6. Score=0.5353 | This course does not recommend Python for beginners who only want native mobile programming.
7. Score=0.5335 | This tutorial is not about Python programming for Android development.
8. Score=0.5302 | BeeWare lets developers build desktop and mobile apps in Python from one codebase.
9. Score=0.5096 | Mobile developers do not usually pick Python as their first choice for native iPhone apps.
10. Score=0.4532 | Kivy is a 

## Step 5: Stage 2 Reranking with Cohere

In [13]:
# Rerank the top 10 dense-retrieval results with Cohere.
cohere_client = cohere.Client(api_key=COHERE_API_KEY)
rerank_response = cohere_client.rerank(
    model="rerank-english-v3.0",
    query=query,
    documents=top_10_documents,
    top_n=10,
)

print("AFTER RERANKING")
for rank, result in enumerate(rerank_response.results, start=1):
    print(f"{rank}. Score={result.relevance_score:.4f} | {top_10_documents[result.index]}")


AFTER RERANKING
1. Score=0.9819 | Python is not suitable for mobile apps because most deployments target desktop or server workflows.
2. Score=0.9604 | Mobile developers do not usually pick Python as their first choice for native iPhone apps.
3. Score=0.9308 | Python frameworks are not the standard route for building high performance mobile interfaces.
4. Score=0.9052 | This tutorial is not about Python programming for Android development.
5. Score=0.8852 | BeeWare is interesting, but Python is not yet the default option for mobile development teams.
6. Score=0.8303 | This course does not recommend Python for beginners who only want native mobile programming.
7. Score=0.7942 | BeeWare lets developers build desktop and mobile apps in Python from one codebase.
8. Score=0.7658 | The article explains why Python is not commonly used for native mobile game engines.
9. Score=0.6589 | Python does not dominate the mobile app market the way it does in data science.
10. Score=0.4176 | Kivy is a P

## Step 6: Comparison Table

In [14]:
# Compare positions before and after reranking and flag big moves.
after_reranking_documents = [top_10_documents[result.index] for result in rerank_response.results]
after_rank_lookup = {document: rank for rank, document in enumerate(after_reranking_documents, start=1)}

print("| Rank | Before Reranking | After Reranking |")
print("|---:|---|---|")
for rank in range(1, 11):
    before_document = top_10_documents[rank - 1]
    after_document = after_reranking_documents[rank - 1]
    before_document_note = ""
    if abs(rank - after_rank_lookup[before_document]) >= 3:
        before_document_note = " [Moved significantly]"
    print(f"| {rank} | {before_document}{before_document_note} | {after_document} |")

print("\nObservations:")
negation_in_top_10 = [document for document in top_10_documents if document in negation_documents]
ambiguous_in_top_10 = [document for document in top_10_documents if document in ambiguous_documents]

if negation_in_top_10:
    print("- Negation-heavy results appeared in dense retrieval and were candidates for demotion after reranking.")
if ambiguous_in_top_10:
    print("- Ambiguous results appeared in dense retrieval and were candidates for reordering after reranking.")
if not negation_in_top_10 and not ambiguous_in_top_10:
    print("- Dense retrieval already surfaced mostly relevant programming results, so reranking mainly fine-tuned the order.")


| Rank | Before Reranking | After Reranking |
|---:|---|---|
| 1 | Python is not suitable for mobile apps because most deployments target desktop or server workflows. | Python is not suitable for mobile apps because most deployments target desktop or server workflows. |
| 2 | Python frameworks are not the standard route for building high performance mobile interfaces. | Mobile developers do not usually pick Python as their first choice for native iPhone apps. |
| 3 | Python does not dominate the mobile app market the way it does in data science. [Moved significantly] | Python frameworks are not the standard route for building high performance mobile interfaces. |
| 4 | BeeWare is interesting, but Python is not yet the default option for mobile development teams. | This tutorial is not about Python programming for Android development. |
| 5 | The article explains why Python is not commonly used for native mobile game engines. [Moved significantly] | BeeWare is interesting, but Python is

## Step 7: Reflection Questions

Q1: Why does dense retrieval fail for complex or ambiguous queries?

A1: Dense retrieval compresses meaning into a single vector, so it can overvalue broad keyword overlap while missing negation, subtle intent, or word-sense differences. That makes it easier for partially related but incorrect documents to rank highly.



Q2: How does reranking improve result ordering?

A2: Reranking looks at the query and each candidate document together, so it can judge relevance with more context than a single embedding score. That usually helps push truly relevant results upward and move misleading matches downward.



Q3: Why is Cross-Encoder attention more accurate than simple vector similarity?

A3: A cross-encoder compares both texts token by token in the same forward pass, which lets it notice interactions like negation, phrasing, and exact intent. Simple vector similarity is faster, but it loses that fine-grained comparison when everything is reduced to fixed embeddings.



Q4: Why is it impractical to run Cross-Encoders across the entire dataset?

A4: Cross-encoders are much more expensive because they must score the query against each document pair individually. That becomes too slow and costly at scale, which is why they are usually used only on a small shortlist returned by a faster dense retriever.



